In [0]:
# ============================================================
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
# MODULE 4 — PRICING ELASTICITY & MARKET INTELLIGENCE
# Author: Byron Kamau
# ============================================================
# DEPENDS ON: Module 7, Module 1, Module 2, Module 3
#
# OUTPUTS:
#   gold_price_elasticity       — elasticity per category
#   gold_competitor_pricing     — your RRP vs market index
#   gold_whitespace             — underpenetrated regions/channels
#   gold_pricing_recommendations — structured action recommendations
# ============================================================

# Module 4 — Pricing Elasticity & Market Intelligence
**Layer:** Silver → Gold

**What this notebook does:**
1. Calculates price elasticity of demand per category using OLS regression
2. Classifies SKUs as elastic, inelastic, or unit-elastic
3. Builds competitor pricing index from synthetic market data
4. Identifies whitespace — underpenetrated regions and channels
5. Surfaces structured pricing and channel recommendations

## Cell 1 — Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

DATABASE = "uk_retail_commercial"
spark.sql(f"USE {DATABASE}")

print("✅ Database set:", DATABASE)

✅ Database set: uk_retail_commercial


## Cell 2 — Load Source Data

In [0]:
df_silver = spark.table(f"{DATABASE}.silver_pos_cleaned")
df_erp    = spark.table(f"{DATABASE}.silver_erp_product_master")
df_fund   = spark.table(f"{DATABASE}.silver_retailer_funding")

print(f"silver_pos_cleaned : {df_silver.count():,} rows")
print(f"silver_erp_product_master : {df_erp.count()} SKUs")

silver_pos_cleaned : 958,501 rows
silver_erp_product_master : 40 SKUs


## Cell 3 — Build Weekly Price-Volume Data for Elasticity
Aggregate to weekly average price and total units per category
This gives us the price-volume relationship needed for OLS regression

In [0]:
# Weekly average price and volume by category
df_pv = df_silver \
    .withColumn("week_start", F.date_trunc("week", F.col("invoice_date"))) \
    .groupBy("category", "week_start") \
    .agg(
        F.avg("unit_price_gbp").alias("avg_price"),
        F.sum("quantity").alias("total_units"),
        F.sum("gross_revenue_gbp").alias("total_revenue"),
        F.sum("contribution_margin_gbp").alias("total_margin")
    ) \
    .filter(
        F.col("avg_price").isNotNull() &
        (F.col("avg_price") > 0) &
        F.col("total_units").isNotNull() &
        (F.col("total_units") > 0)
    )

df_pv_pd = df_pv.toPandas()
df_pv_pd["week_start"] = pd.to_datetime(df_pv_pd["week_start"])

categories = df_pv_pd["category"].unique().tolist()
print(f"✅ Categories for elasticity: {categories}")
print(f"✅ Total price-volume rows  : {len(df_pv_pd):,}")
display(df_pv.limit(10))

✅ Categories for elasticity: ['Travel', 'Bathing', 'Nursery', 'Other', 'Clothing', 'Toys', 'Sleep', 'Feeding', 'Healthcare']
✅ Total price-volume rows  : 936


category,week_start,avg_price,total_units,total_revenue,total_margin
Travel,2011-11-14T00:00:00.000Z,3.3978060046189316,15151,36659.60000000019,19152.990000000056
Travel,2010-11-15T00:00:00.000Z,3.540336842105249,13590,33152.4500000001,17404.410000000025
Bathing,2010-11-22T00:00:00.000Z,3.9327178303519643,13375,26764.440000000122,13627.140000000016
Nursery,2010-11-29T00:00:00.000Z,3.1199557195571876,20186,38005.43000000005,18356.89999999998
Nursery,2010-10-18T00:00:00.000Z,3.1493918918918813,11076,19933.880000000016,9032.480000000005
Nursery,2010-11-01T00:00:00.000Z,3.159835958005238,13965,22728.71000000009,10677.810000000023
Other,2010-11-15T00:00:00.000Z,5.136287261475002,12638,31924.770000000008,13217.719999999983
Clothing,2010-10-11T00:00:00.000Z,2.7066437499999862,20871,34783.04000000006,15179.699999999937
Other,2010-10-25T00:00:00.000Z,4.444180392156849,10007,23104.14000000003,8761.110000000026
Toys,2011-11-14T00:00:00.000Z,3.1541122504047396,14429,29295.350000000202,14440.299999999996


## Cell 4 — OLS Price Elasticity Regression
log(Quantity) = α + β × log(Price)
β = Price Elasticity of Demand
β < -1 = Elastic (price sensitive)
β between -1 and 0 = Inelastic
β > 0 = Unusual / Giffen good

In [0]:
elasticity_results = []

for category in categories:
    df_cat = df_pv_pd[df_pv_pd["category"] == category].copy()

    # Need at least 10 data points for meaningful regression
    if len(df_cat) < 10:
        print(f"  ⚠️ Skipping {category} — insufficient data ({len(df_cat)} rows)")
        continue

    # Log-log transformation
    df_cat["log_price"] = np.log(df_cat["avg_price"])
    df_cat["log_units"] = np.log(df_cat["total_units"])

    # Remove infinities or nulls
    df_cat = df_cat.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["log_price", "log_units"]
    )

    if len(df_cat) < 10:
        continue

    # OLS Regression: log(Q) ~ log(P)
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        df_cat["log_price"], df_cat["log_units"]
    )

    elasticity = round(slope, 3)
    r_squared  = round(r_value ** 2, 3)

    # Classification
    if elasticity < -1.5:
        classification = "HIGHLY ELASTIC"
        pricing_action = "Price cut opportunity — volume gain outweighs margin loss"
    elif elasticity < -1.0:
        classification = "ELASTIC"
        pricing_action = "Cautious on price increases — consider promotional investment"
    elif elasticity < -0.5:
        classification = "MILDLY ELASTIC"
        pricing_action = "Moderate price sensitivity — test small price changes"
    elif elasticity < 0:
        classification = "INELASTIC"
        pricing_action = "Price increase opportunity — volume loss minimal"
    else:
        classification = "UNUSUAL / REVIEW"
        pricing_action = "Review data — positive elasticity is unexpected"

    # Average price and units for context
    avg_price = round(df_cat["avg_price"].mean(), 2)
    avg_units = round(df_cat["total_units"].mean(), 0)
    avg_revenue = round(df_cat["total_revenue"].mean(), 2)

    elasticity_results.append({
        "category":          category,
        "elasticity_coeff":  elasticity,
        "r_squared":         r_squared,
        "p_value":           round(p_value, 4),
        "std_error":         round(std_err, 4),
        "classification":    classification,
        "pricing_action":    pricing_action,
        "avg_weekly_price":  avg_price,
        "avg_weekly_units":  avg_units,
        "avg_weekly_revenue": avg_revenue,
        "data_points":       len(df_cat)
    })

    print(f"  ✅ {category:<15} elasticity: {elasticity:>7.3f}  "
          f"R²: {r_squared:.3f}  {classification}")

print(f"\n✅ Elasticity calculated for {len(elasticity_results)} categories")

  ✅ Travel          elasticity:  -1.021  R²: 0.050  ELASTIC
  ✅ Bathing         elasticity:  -1.305  R²: 0.165  ELASTIC
  ✅ Nursery         elasticity:  -1.581  R²: 0.109  HIGHLY ELASTIC
  ✅ Other           elasticity:  -0.041  R²: 0.002  INELASTIC
  ✅ Clothing        elasticity:  -1.011  R²: 0.080  ELASTIC
  ✅ Toys            elasticity:  -1.667  R²: 0.154  HIGHLY ELASTIC
  ✅ Sleep           elasticity:  -1.309  R²: 0.120  ELASTIC
  ✅ Feeding         elasticity:  -2.025  R²: 0.117  HIGHLY ELASTIC
  ✅ Healthcare      elasticity:  -1.557  R²: 0.149  HIGHLY ELASTIC

✅ Elasticity calculated for 9 categories


## Cell 5 — Write Gold: Price Elasticity Table

In [0]:
df_elasticity = spark.createDataFrame(pd.DataFrame(elasticity_results))
df_elasticity = df_elasticity.withColumn("calculated_at", F.current_timestamp())

(df_elasticity.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_price_elasticity"))

print(f"✅ gold_price_elasticity written: {df_elasticity.count()} categories")
display(df_elasticity.orderBy("elasticity_coeff"))

✅ gold_price_elasticity written: 9 categories


category,elasticity_coeff,r_squared,p_value,std_error,classification,pricing_action,avg_weekly_price,avg_weekly_units,avg_weekly_revenue,data_points,calculated_at
Feeding,-2.025,0.117,4.0E-4,0.552,HIGHLY ELASTIC,Price cut opportunity — volume gain outweighs margin loss,3.46,10765.0,20794.85,104,2026-03-20T11:01:02.687Z
Toys,-1.667,0.154,0.0,0.3865,HIGHLY ELASTIC,Price cut opportunity — volume gain outweighs margin loss,3.43,9313.0,14794.43,104,2026-03-20T11:01:02.687Z
Nursery,-1.581,0.109,6.0E-4,0.4486,HIGHLY ELASTIC,Price cut opportunity — volume gain outweighs margin loss,3.07,9755.0,16501.85,104,2026-03-20T11:01:02.687Z
Healthcare,-1.557,0.149,1.0E-4,0.3682,HIGHLY ELASTIC,Price cut opportunity — volume gain outweighs margin loss,3.5,10150.0,19464.36,104,2026-03-20T11:01:02.687Z
Sleep,-1.309,0.12,3.0E-4,0.3518,ELASTIC,Cautious on price increases — consider promotional investment,3.62,12043.0,22350.97,104,2026-03-20T11:01:02.687Z
Bathing,-1.305,0.165,0.0,0.2902,ELASTIC,Cautious on price increases — consider promotional investment,3.58,8838.0,16676.37,104,2026-03-20T11:01:02.687Z
Travel,-1.021,0.05,0.0224,0.4401,ELASTIC,Cautious on price increases — consider promotional investment,3.55,7856.0,16696.86,104,2026-03-20T11:01:02.687Z
Clothing,-1.011,0.08,0.0036,0.3395,ELASTIC,Cautious on price increases — consider promotional investment,2.93,11356.0,17801.26,104,2026-03-20T11:01:02.687Z
Other,-0.041,0.002,0.6429,0.0873,INELASTIC,Price increase opportunity — volume loss minimal,8.52,10148.0,26755.38,104,2026-03-20T11:01:02.687Z


## Cell 6 — Build Competitor Pricing Index
Synthetic competitor data simulating UK nursery market pricing
Compares your brand RRP vs competitor price points by category

In [0]:
import pandas as pd
import numpy as np

np.random.seed(99)

# Get our average selling prices by category from Silver
df_our_prices = df_silver.groupBy("category").agg(
    F.avg("unit_price_gbp").alias("our_avg_price"),
    F.sum("gross_revenue_gbp").alias("our_total_revenue")
).toPandas()

# Generate synthetic competitor pricing
competitors = ["MamaMoo", "BabyBright", "NurtureUK", "TinySteps", "PramWorld"]

comp_rows = []
for _, row in df_our_prices.iterrows():
    category    = row["category"]
    our_price   = row["our_avg_price"]
    our_revenue = row["our_total_revenue"]

    for comp in competitors:
        # Competitor price as a % variance from our price
        price_variance = np.random.uniform(-0.25, 0.30)
        comp_price = round(our_price * (1 + price_variance), 2)

        # Competitor promo depth (how deep they discount)
        promo_depth = round(np.random.uniform(10, 35), 1)

        # Market share index (simulated)
        market_share = round(np.random.uniform(5, 25), 1)

        comp_rows.append({
            "category":              category,
            "competitor":            comp,
            "our_avg_price_gbp":     round(our_price, 2),
            "competitor_price_gbp":  comp_price,
            "price_index":           round(comp_price / our_price * 100, 1),
            "price_gap_gbp":         round(comp_price - our_price, 2),
            "price_gap_pct":         round((comp_price - our_price) / our_price * 100, 1),
            "competitor_promo_depth_pct": promo_depth,
            "competitor_market_share_pct": market_share,
        })

df_comp_pd = pd.DataFrame(comp_rows)

# Add pricing position flag
df_comp_pd["pricing_position"] = df_comp_pd["price_gap_pct"].apply(
    lambda x: "WE ARE PREMIUM" if x < -10
    else ("PRICE PARITY" if abs(x) <= 10
    else "COMPETITOR PREMIUM")
)

# Add risk flag
df_comp_pd["competitive_risk"] = df_comp_pd.apply(
    lambda r: "HIGH RISK" if (r["price_gap_pct"] < -15 and r["competitor_market_share_pct"] > 15)
    else ("MODERATE RISK" if r["price_gap_pct"] < -5
    else "LOW RISK"), axis=1
)

df_competitor = spark.createDataFrame(df_comp_pd)
df_competitor = df_competitor.withColumn("calculated_at", F.current_timestamp())

(df_competitor.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_competitor_pricing"))

print(f"✅ gold_competitor_pricing written: {df_competitor.count()} rows")
display(df_competitor.orderBy("category", "competitor"))

✅ gold_competitor_pricing written: 45 rows


category,competitor,our_avg_price_gbp,competitor_price_gbp,price_index,price_gap_gbp,price_gap_pct,competitor_promo_depth_pct,competitor_market_share_pct,pricing_position,competitive_risk,calculated_at
Bathing,BabyBright,3.53,2.94,83.4,-0.59,-16.6,21.8,15.9,WE ARE PREMIUM,HIGH RISK,2026-03-20T11:01:07.321Z
Bathing,MamaMoo,3.53,3.99,113.2,0.46,13.2,30.7,23.4,COMPETITOR PREMIUM,LOW RISK,2026-03-20T11:01:07.321Z
Bathing,NurtureUK,3.53,3.65,103.5,0.12,3.5,28.0,21.5,PRICE PARITY,LOW RISK,2026-03-20T11:01:07.321Z
Bathing,PramWorld,3.53,3.12,88.5,-0.41,-11.5,30.7,14.3,WE ARE PREMIUM,MODERATE RISK,2026-03-20T11:01:07.321Z
Bathing,TinySteps,3.53,3.1,87.9,-0.43,-12.1,12.6,17.3,WE ARE PREMIUM,MODERATE RISK,2026-03-20T11:01:07.321Z
Clothing,BabyBright,2.93,2.32,79.3,-0.61,-20.7,15.9,5.1,WE ARE PREMIUM,MODERATE RISK,2026-03-20T11:01:07.321Z
Clothing,MamaMoo,2.93,2.43,83.0,-0.5,-17.0,15.3,14.5,WE ARE PREMIUM,MODERATE RISK,2026-03-20T11:01:07.321Z
Clothing,NurtureUK,2.93,3.64,124.4,0.71,24.4,23.8,8.4,COMPETITOR PREMIUM,LOW RISK,2026-03-20T11:01:07.321Z
Clothing,PramWorld,2.93,3.04,103.9,0.11,3.9,26.0,21.0,PRICE PARITY,LOW RISK,2026-03-20T11:01:07.321Z
Clothing,TinySteps,2.93,3.69,126.1,0.76,26.1,23.6,5.8,COMPETITOR PREMIUM,LOW RISK,2026-03-20T11:01:07.321Z


## Cell 7 — Whitespace Analysis
Identify underpenetrated UK regions and channels
vs. what we would expect given population distribution

In [0]:
# UK region population weights (approximate % of UK population)
uk_population_weights = {
    "London":           13.4,
    "South East":       14.0,
    "North West":       10.7,
    "East of England":   9.4,
    "West Midlands":     8.7,
    "South West":        8.7,
    "Yorkshire":         8.4,
    "East Midlands":     7.5,
    "North East":        4.0,
    "Scotland":          8.3,
    "Wales":             4.9,
    "Northern Ireland":  2.9,
    "Other UK":          0.0,
}

# Actual revenue share by region
df_region_share = df_silver.groupBy("uk_region").agg(
    F.sum("gross_revenue_gbp").alias("region_revenue"),
    F.sum("quantity").alias("region_units"),
    F.countDistinct("customer_id").alias("region_customers")
)

total_revenue = df_silver.agg(
    F.sum("gross_revenue_gbp").alias("total")
).collect()[0]["total"]

df_region_share = df_region_share.withColumn(
    "revenue_share_pct",
    F.round(F.col("region_revenue") / F.lit(total_revenue) * 100, 2)
)

df_region_pd = df_region_share.toPandas()

# Add population weight and calculate whitespace gap
df_region_pd["population_weight_pct"] = df_region_pd["uk_region"].map(
    uk_population_weights
).fillna(0)

df_region_pd["whitespace_gap_pct"] = (
    df_region_pd["population_weight_pct"] -
    df_region_pd["revenue_share_pct"]
).round(2)

df_region_pd["whitespace_flag"] = df_region_pd["whitespace_gap_pct"].apply(
    lambda x: "MAJOR WHITESPACE" if x > 5
    else ("WHITESPACE" if x > 2
    else ("ON TRACK" if x >= -1
    else "OVER-INDEXED"))
)

df_region_pd["recommended_action"] = df_region_pd.apply(
    lambda r: f"Increase distribution and trade investment in {r['uk_region']}"
    if r["whitespace_flag"] in ["MAJOR WHITESPACE", "WHITESPACE"]
    else (f"Maintain current strategy in {r['uk_region']}"
    if r["whitespace_flag"] == "ON TRACK"
    else f"Optimise spend efficiency in {r['uk_region']} — already over-indexed"),
    axis=1
)

# Channel whitespace
df_channel_share = df_silver.groupBy("channel").agg(
    F.sum("gross_revenue_gbp").alias("channel_revenue"),
    F.avg("contribution_margin_pct").alias("channel_avg_margin_pct")
).withColumn(
    "channel_revenue_share_pct",
    F.round(F.col("channel_revenue") / F.lit(total_revenue) * 100, 1)
)

df_whitespace = spark.createDataFrame(df_region_pd)
df_whitespace = df_whitespace.withColumn("calculated_at", F.current_timestamp())

(df_whitespace.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_whitespace"))

print(f"✅ gold_whitespace written: {df_whitespace.count()} regions")
display(df_whitespace.orderBy(F.col("whitespace_gap_pct").desc()))

print("\n📊 Channel Revenue Split:")
display(df_channel_share)

✅ gold_whitespace written: 11 regions


uk_region,region_revenue,region_units,region_customers,revenue_share_pct,population_weight_pct,whitespace_gap_pct,whitespace_flag,recommended_action,calculated_at
London,661450.9799999999,432007,183,3.7,13.4,9.7,MAJOR WHITESPACE,Increase distribution and trade investment in London,2026-03-20T11:01:12.646Z
South East,1304104.4600000023,906090,481,7.3,14.0,6.7,MAJOR WHITESPACE,Increase distribution and trade investment in South East,2026-03-20T11:01:12.646Z
North West,1390482.1399999987,851528,486,7.78,10.7,2.92,WHITESPACE,Increase distribution and trade investment in North West,2026-03-20T11:01:12.646Z
East of England,1194621.66,697009,494,6.68,9.4,2.72,WHITESPACE,Increase distribution and trade investment in East of England,2026-03-20T11:01:12.646Z
Yorkshire,1036882.3500000006,584730,478,5.8,8.4,2.6,WHITESPACE,Increase distribution and trade investment in Yorkshire,2026-03-20T11:01:12.646Z
West Midlands,1325988.2999999993,758557,489,7.42,8.7,1.28,ON TRACK,Maintain current strategy in West Midlands,2026-03-20T11:01:12.646Z
South West,1379220.7299999981,811946,493,7.72,8.7,0.98,ON TRACK,Maintain current strategy in South West,2026-03-20T11:01:12.646Z
Scotland,1342184.3799999987,747611,492,7.51,8.3,0.79,ON TRACK,Maintain current strategy in Scotland,2026-03-20T11:01:12.646Z
Wales,1251893.1800000006,789842,490,7.01,4.9,-2.11,OVER-INDEXED,Optimise spend efficiency in Wales — already over-indexed,2026-03-20T11:01:12.646Z
North East,1279067.9500000016,699682,489,7.16,4.0,-3.16,OVER-INDEXED,Optimise spend efficiency in North East — already over-indexed,2026-03-20T11:01:12.646Z



📊 Channel Revenue Split:


channel,channel_revenue,channel_avg_margin_pct,channel_revenue_share_pct
DTC,8336195.539999308,63.33848744853463,46.6
Retail,9534782.21999902,26.773905136763688,53.4


## Cell 8 — Generate Pricing Recommendations
Structured recommendations based on elasticity + competitor + whitespace

In [0]:
recommendations = []

# Pull elasticity data
df_elas_pd = spark.table(f"{DATABASE}.gold_price_elasticity").toPandas()

# Pull competitor pricing
df_comp_agg = spark.table(f"{DATABASE}.gold_competitor_pricing") \
    .groupBy("category", "pricing_position", "competitive_risk") \
    .agg(
        F.avg("price_gap_pct").alias("avg_price_gap_pct"),
        F.avg("competitor_promo_depth_pct").alias("avg_comp_promo_depth")
    ).toPandas()

# Pull whitespace
df_ws_pd = spark.table(f"{DATABASE}.gold_whitespace") \
    .filter(F.col("whitespace_flag").isin(["MAJOR WHITESPACE", "WHITESPACE"])) \
    .toPandas()

# ── RECOMMENDATION 1: Pricing by elasticity ────────────────
for _, row in df_elas_pd.iterrows():
    if row["classification"] == "INELASTIC":
        recommendations.append({
            "rec_id":         f"REC-P-{len(recommendations)+1:03d}",
            "rec_type":       "PRICING",
            "category":       row["category"],
            "issue":          f"{row['category']} is price inelastic (elasticity: {row['elasticity_coeff']})",
            "evidence":       f"R²={row['r_squared']} — price changes have limited volume impact",
            "action":         f"Test 5-8% RRP increase on {row['category']} — minimal volume loss expected",
            "commercial_impact": "Margin improvement without significant volume loss",
            "priority":       "HIGH"
        })
    elif row["classification"] in ["HIGHLY ELASTIC", "ELASTIC"]:
        recommendations.append({
            "rec_id":         f"REC-P-{len(recommendations)+1:03d}",
            "rec_type":       "PRICING",
            "category":       row["category"],
            "issue":          f"{row['category']} is price elastic (elasticity: {row['elasticity_coeff']})",
            "evidence":       f"R²={row['r_squared']} — price increases drive significant volume loss",
            "action":         f"Hold or reduce price on {row['category']} — focus on promotional investment",
            "commercial_impact": "Protect volume and market share",
            "priority":       "MEDIUM"
        })

# ── RECOMMENDATION 2: Competitive pricing gaps ─────────────
high_risk = df_comp_agg[df_comp_agg["competitive_risk"] == "HIGH RISK"]
for _, row in high_risk.iterrows():
    recommendations.append({
        "rec_id":         f"REC-C-{len(recommendations)+1:03d}",
        "rec_type":       "COMPETITIVE",
        "category":       row["category"],
        "issue":          f"Competitors pricing {abs(row['avg_price_gap_pct']):.1f}% below us in {row['category']}",
        "evidence":       f"Average competitor promo depth: {row['avg_comp_promo_depth']:.1f}%",
        "action":         f"Review RRP architecture for {row['category']} — consider targeted price match on hero SKUs",
        "commercial_impact": "Defend volume against competitive undercutting",
        "priority":       "HIGH"
    })

# ── RECOMMENDATION 3: Whitespace regions ───────────────────
for _, row in df_ws_pd.iterrows():
    recommendations.append({
        "rec_id":         f"REC-W-{len(recommendations)+1:03d}",
        "rec_type":       "WHITESPACE",
        "category":       "All Categories",
        "issue":          f"{row['uk_region']} is underpenetrated — {row['whitespace_gap_pct']}% gap vs population weight",
        "evidence":       f"Current revenue share: {row['revenue_share_pct']}% vs population: {row['population_weight_pct']}%",
        "action":         f"Increase retailer distribution and targeted trade investment in {row['uk_region']}",
        "commercial_impact": "Unlock incremental revenue from untapped regional demand",
        "priority":       "MEDIUM" if row["whitespace_gap_pct"] < 5 else "HIGH"
    })

df_recs = spark.createDataFrame(pd.DataFrame(recommendations))
df_recs = df_recs.withColumn("created_at", F.current_timestamp())

(df_recs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_pricing_recommendations"))

print(f"✅ gold_pricing_recommendations written: {df_recs.count()} recommendations")
display(df_recs.orderBy("priority", "rec_type"))

✅ gold_pricing_recommendations written: 16 recommendations


rec_id,rec_type,category,issue,evidence,action,commercial_impact,priority,created_at
REC-C-010,COMPETITIVE,Healthcare,Competitors pricing 24.0% below us in Healthcare,Average competitor promo depth: 29.7%,Review RRP architecture for Healthcare — consider targeted price match on hero SKUs,Defend volume against competitive undercutting,HIGH,2026-03-20T11:01:19.276Z
REC-C-011,COMPETITIVE,Bathing,Competitors pricing 16.6% below us in Bathing,Average competitor promo depth: 21.8%,Review RRP architecture for Bathing — consider targeted price match on hero SKUs,Defend volume against competitive undercutting,HIGH,2026-03-20T11:01:19.276Z
REC-P-004,PRICING,Other,Other is price inelastic (elasticity: -0.041),R²=0.002 — price changes have limited volume impact,Test 5-8% RRP increase on Other — minimal volume loss expected,Margin improvement without significant volume loss,HIGH,2026-03-20T11:01:19.276Z
REC-W-013,WHITESPACE,All Categories,London is underpenetrated — 9.7% gap vs population weight,Current revenue share: 3.7% vs population: 13.4%,Increase retailer distribution and targeted trade investment in London,Unlock incremental revenue from untapped regional demand,HIGH,2026-03-20T11:01:19.276Z
REC-W-016,WHITESPACE,All Categories,South East is underpenetrated — 6.7% gap vs population weight,Current revenue share: 7.3% vs population: 14.0%,Increase retailer distribution and targeted trade investment in South East,Unlock incremental revenue from untapped regional demand,HIGH,2026-03-20T11:01:19.276Z
REC-P-001,PRICING,Travel,Travel is price elastic (elasticity: -1.021),R²=0.05 — price increases drive significant volume loss,Hold or reduce price on Travel — focus on promotional investment,Protect volume and market share,MEDIUM,2026-03-20T11:01:19.276Z
REC-P-002,PRICING,Bathing,Bathing is price elastic (elasticity: -1.305),R²=0.165 — price increases drive significant volume loss,Hold or reduce price on Bathing — focus on promotional investment,Protect volume and market share,MEDIUM,2026-03-20T11:01:19.276Z
REC-P-003,PRICING,Nursery,Nursery is price elastic (elasticity: -1.581),R²=0.109 — price increases drive significant volume loss,Hold or reduce price on Nursery — focus on promotional investment,Protect volume and market share,MEDIUM,2026-03-20T11:01:19.276Z
REC-P-005,PRICING,Clothing,Clothing is price elastic (elasticity: -1.011),R²=0.08 — price increases drive significant volume loss,Hold or reduce price on Clothing — focus on promotional investment,Protect volume and market share,MEDIUM,2026-03-20T11:01:19.276Z
REC-P-006,PRICING,Toys,Toys is price elastic (elasticity: -1.667),R²=0.154 — price increases drive significant volume loss,Hold or reduce price on Toys — focus on promotional investment,Protect volume and market share,MEDIUM,2026-03-20T11:01:19.276Z


## Cell 9 — Module 4 Summary & Quality Check

In [0]:
print("=" * 65)
print("  MODULE 4 — QUALITY CHECK & MARKET INTELLIGENCE SUMMARY")
print("=" * 65)

tables = [
    "gold_price_elasticity",
    "gold_competitor_pricing",
    "gold_whitespace",
    "gold_pricing_recommendations",
]

for t in tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.{t}").collect()[0]["n"]
    print(f"  ✅  {t:<35} {n:>8} rows")

print("=" * 65)

# Elasticity summary
print()
print("  📊 PRICE ELASTICITY BY CATEGORY")
print("  " + "-" * 55)
spark.sql(f"""
    SELECT category, elasticity_coeff, classification, pricing_action
    FROM {DATABASE}.gold_price_elasticity
    ORDER BY elasticity_coeff
""").show(truncate=False)

# Whitespace summary
print("  🗺️  WHITESPACE OPPORTUNITIES")
print("  " + "-" * 55)
spark.sql(f"""
    SELECT uk_region, revenue_share_pct,
           population_weight_pct, whitespace_gap_pct, whitespace_flag
    FROM {DATABASE}.gold_whitespace
    WHERE whitespace_flag IN ('MAJOR WHITESPACE','WHITESPACE')
    ORDER BY whitespace_gap_pct DESC
""").show(truncate=False)

# Recommendations summary
print("  🎯 RECOMMENDATIONS BY TYPE & PRIORITY")
print("  " + "-" * 55)
spark.sql(f"""
    SELECT rec_type, priority, COUNT(*) AS count
    FROM {DATABASE}.gold_pricing_recommendations
    GROUP BY rec_type, priority
    ORDER BY priority, rec_type
""").show(truncate=False)

print("=" * 65)
print("  🎉 MODULE 4 COMPLETE!")
print("  👉 Next: Module 6 — Recommendations Engine")

  MODULE 4 — QUALITY CHECK & MARKET INTELLIGENCE SUMMARY
  ✅  gold_price_elasticity                      9 rows
  ✅  gold_competitor_pricing                   45 rows
  ✅  gold_whitespace                           11 rows
  ✅  gold_pricing_recommendations              16 rows

  📊 PRICE ELASTICITY BY CATEGORY
  -------------------------------------------------------
+----------+----------------+--------------+-------------------------------------------------------------+
|category  |elasticity_coeff|classification|pricing_action                                               |
+----------+----------------+--------------+-------------------------------------------------------------+
|Feeding   |-2.025          |HIGHLY ELASTIC|Price cut opportunity — volume gain outweighs margin loss    |
|Toys      |-1.667          |HIGHLY ELASTIC|Price cut opportunity — volume gain outweighs margin loss    |
|Nursery   |-1.581          |HIGHLY ELASTIC|Price cut opportunity — volume gain outweighs margin